In [1]:
import os
import json
import requests
import sqlite3
from datetime import datetime

# Project Configuration
FOLDER_NAME = r"D:\DE_AI_CodeReview\Final_Code_Review_System"
GROQ_API_KEY = "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"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

# Database path (inside the new project folder)
DB_PATH = os.path.join(FOLDER_NAME, "execution_tracking.db")

# Reports folder (inside the new project folder)
REPORTS_DIR = os.path.join(FOLDER_NAME, "reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f" Project Folder: {FOLDER_NAME}")
print(f" Reports Folder: {REPORTS_DIR}")
print(" Configuration loaded successfully!")

 Project Folder: D:\DE_AI_CodeReview\Final_Code_Review_System
 Reports Folder: D:\DE_AI_CodeReview\Final_Code_Review_System\reports
 Configuration loaded successfully!


In [2]:
def detect_file_type(file_path):
    """
    Detects file type based on extension.
    Supports: .py (python), .sql (sql), .ipynb (jupyter), else unknown.
    """
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".py":
        return "python"
    elif ext == ".sql":
        return "sql"
    elif ext == ".ipynb":
        return "jupyter"
    else:
        return "unknown"


def extract_code_from_ipynb(file_path):
    """
    Extracts all Python code from a Jupyter notebook (.ipynb) file.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
        
        code_cells = []
        for cell in notebook.get("cells", []):
            if cell.get("cell_type") == "code":
                code_cells.append("".join(cell.get("source", [])))
        
        return "\n\n# --- Next Cell ---\n\n".join(code_cells)
    
    except Exception as e:
        print(f" Error parsing Jupyter notebook: {e}")
        return None


def read_code_file(file_path):
    """
    Reads any supported code file (.py, .sql, .ipynb) and returns content as string.
    """
    file_type = detect_file_type(file_path)
    
    # Handle Jupyter notebooks specially
    if file_type == "jupyter":
        return extract_code_from_ipynb(file_path)
    
    # Handle regular text files
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f" File not found: {file_path}")
        return None
    except Exception as e:
        print(f" Error reading file: {e}")
        return None

In [3]:
def build_prompt(code_content, file_type="python"):
    """
    Builds a professional, detailed prompt for the AI.
    Includes security, performance, and maintainability checklists.
    """
    base_instruction = """
You are a Senior Data Engineer with 10+ years of experience in production-grade systems.
Review the following code thoroughly. Provide a structured analysis covering:
1. **Bug Analysis**: Identify logical errors, performance bottlenecks (memory/time), and edge cases.
2. **Security & Reliability**: Check for SQL injection, insecure data handling, or unhandled exceptions.
3. **Code Maintainability**: Point out code duplication, naming issues, or violation of SOLID/DRY principles.
4. **Corrected Code**: Provide the fully refactored, production-ready code inside a code block.

Be concise but thorough. Focus on what actually matters in a data engineering context.
"""
    
    if file_type == "python":
        return f"""
{base_instruction}

Language: Python
Goal: Optimize for large-scale data processing (Pandas, Polars, or native Python).

Buggy Code:
{code_content}

Response Format:
- Bug Analysis: (List issues with explanations)
- Security & Reliability: (Any risks?)
- Corrected Code: (Complete fixed code)
"""
    
    elif file_type == "sql":
        return f"""
{base_instruction}

Language: SQL (PostgreSQL/Standard)
Goal: Optimize query performance, avoid Cartesian products, and ensure proper indexing hints.

Buggy SQL:
{code_content}

Response Format:
- Bug Analysis: (Explain missing joins, inefficient filters, or full scans)
- Corrected Code: (Optimized SQL with proper JOINs and WHERE clauses)
"""
    
    elif file_type == "jupyter":
        return f"""
{base_instruction}

Language: Python (Jupyter Notebook)
Goal: Ensure cells run in order, avoid global variable pollution, and improve reproducibility.

Buggy Notebook Code (extracted cells):
{code_content}

Response Format:
- Bug Analysis: (Cell order issues, redundant recomputations, or heavy prints)
- Corrected Code: (Refactored notebook logic as a clean script or well-ordered cells)
"""
    
    else:
        return f"""
{base_instruction}

Language: Unknown
Code:
{code_content}

Please analyze the code and provide the standard format.
"""

In [4]:
def call_groq_api(prompt):
    """
    Sends the prompt to the Groq API and returns the AI's response.
    """
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,  # Low temperature for precise, deterministic results
    }

    try:
        print("📡 Sending request to Groq API...")
        response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()  # Raise an error for bad status codes
        
        result = response.json()
        ai_response = result["choices"][0]["message"]["content"]
        print(" API response received successfully!")
        return ai_response
    
    except requests.exceptions.Timeout:
        print(" API request timed out. Please try again.")
        return None
    except requests.exceptions.RequestException as e:
        print(f" API request failed: {e}")
        return None

In [5]:
def save_report(file_name, ai_analysis, file_type="python"):
    """
    Saves the AI analysis report to a timestamped text file inside the reports folder.
    """
    # Map file type to a clean label
    type_map = {
        "python": "python",
        "sql": "sql",
        "jupyter": "notebook",
        "unknown": "code"
    }
    label = type_map.get(file_type, "code")
    
    # Create a timestamped filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_name = f"ai_review_{label}_{timestamp}.txt"
    report_path = os.path.join(REPORTS_DIR, report_name)
    
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("=" * 70 + "\n")
        f.write(f"  DATA ENGINEERING AI REVIEW REPORT ({file_type.upper()})\n".center(70))
        f.write("=" * 70 + "\n\n")
        f.write(f" Target File: {file_name}\n")
        f.write(f" Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("─" * 70 + "\n")
        f.write(ai_analysis)
        f.write("\n" + "=" * 70 + "\n")
    
    return report_path


def log_to_database(file_name, ai_analysis, status="SUCCESS"):
    """
    Logs the review result into the SQLite database for tracking.
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Create table if it doesn't exist
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS code_review_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_name TEXT NOT NULL,
            execution_status TEXT NOT NULL,
            ai_analysis_result TEXT,
            timestamp TEXT NOT NULL
        )
    """)

    # Insert the log entry
    cursor.execute("""
        INSERT INTO code_review_logs (file_name, execution_status, ai_analysis_result, timestamp)
        VALUES (?, ?, ?, ?)
    """, (
        file_name,
        status,
        ai_analysis,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ))

    conn.commit()
    conn.close()
    print(f" Logged to database: {DB_PATH}")

In [6]:
def analyze_code(file_path):
    """
    The main orchestration function.
    Handles the entire workflow: read -> detect -> prompt -> API -> save -> log.
    """
    # 1. Extract file name and read content
    file_name = os.path.basename(file_path)
    print(f"\n Processing file: {file_name}")
    
    code_content = read_code_file(file_path)
    if code_content is None:
        print(" Aborting: Could not read the file.")
        return
    
    # 2. Detect file type
    file_type = detect_file_type(file_path)
    print(f" Detected type: {file_type}")
    
    # 3. Build the prompt
    prompt = build_prompt(code_content, file_type)
    
    # 4. Call the API
    ai_response = call_groq_api(prompt)
    if ai_response is None:
        print(" Aborting: API call failed.")
        return
    
    # 5. Save the report
    report_path = save_report(file_name, ai_response, file_type)
    print(f" Report saved: {report_path}")
    
    # 6. Log to database
    log_to_database(file_name, ai_response)
    
    # 7. Print a summary preview
    print("\n" + "=" * 55)
    print("         AI CODE REVIEW SUMMARY".center(55))
    print("=" * 55)
    
    # Print first 400 characters as preview
    preview = ai_response[:400]
    if len(ai_response) > 400:
        preview += "... (Full report saved in file)"
    print(preview)
    print("=" * 55)
    print("\n Analysis complete!")

In [7]:
# Example: Analyze a Python file
analyze_code(r"D:\DE_AI_CodeReview\kaggle_real_test.py")

# To analyze a SQL file (uncomment and adjust path):
# analyze_code(r"D:\DE_AI_CodeReview\bug2_sql.py")

# To analyze a Jupyter Notebook (uncomment and adjust path):
# analyze_code(r"D:\DE_AI_CodeReview\Engineering_Code_Review.ipynb")


 Processing file: kaggle_real_test.py
 Detected type: python
📡 Sending request to Groq API...
 API response received successfully!
 Report saved: D:\DE_AI_CodeReview\Final_Code_Review_System\reports\ai_review_python_20260629_125803.txt
 Logged to database: D:\DE_AI_CodeReview\Final_Code_Review_System\execution_tracking.db

                     AI CODE REVIEW SUMMARY            
**Bug Analysis:**

1.  **Inconsistent Date Handling**: The code uses both `pd.to_datetime` and `dt.strftime` to handle dates. It's better to stick with a single method for consistency. In this case, we can use `pd.to_datetime` to convert the date column to datetime format.
2.  **Missing Error Handling**: The code doesn't handle potential errors that may occur during data loading, aggregation, or g... (Full report saved in file)

 Analysis complete!
